# AmSC Compute + Filesystem Smoke Test (via RIG)

This notebook runs the same end-to-end smoke test as [`iri/compute-jobs.ipynb`](../iri/compute-jobs.ipynb), but it authenticates with an **AmSC PAT** and
sends every call **through RIG** (Resource Integration Gateway) instead of calling a facility IRI API directly:

1. **Facilities**: ask RIG which facilities it serves and check which ones your PAT can reach.
2. **Compute**: on the chosen facility, submit a job that writes a timestamped log file containing date/time/hostname/user and a full environment dump.
3. **Filesystem**: list the job directory (pretty table) and download the log file via async FS tasks, then **base64-decode** and print it.

Facility calls go to `<RIG>/rig/external/<facility>/<IRI API path>`. RIG validates the PAT, maps your AmSC project to the
facility project, and attaches the facility credential:

| Tier | Facility credential | What you need |
|---|---|---|
| 1 | Your AmSC identity | Nothing extra |
| 2 | Token RIG obtained through the facility's OAuth (e.g. Globus for ALCF) | Connect the facility once in the Credential Vault |
| 3 | Facility token stored for your project (e.g. NERSC, OLCF) | Save a facility token for this project in the Credential Vault |

## Before you run
Run [`login-amsc.ipynb`](login-amsc.ipynb) to get a PAT (saved to `~/.amsc_token.json`), or set `AMSC_PAT` and `AMSC_RIG_URL`.
RIG uses the PAT's project (`amsc_project_context`) for every call. To use another project, generate a new PAT.

Not every facility is reachable with the PAT alone. See [`amsc-facility-access.ipynb`](amsc-facility-access.ipynb) for what each facility needs
(allocations, Credential Vault setup in RIG-UI).

## 1) Imports & helper functions
Utilities used throughout the notebook.

In [1]:
import os
import sys
import time
import base64
import json
import datetime as dt
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
from dotenv import load_dotenv

load_dotenv()

def die(msg: str) -> None:
    print(f"\nERROR: {msg}")
    raise SystemExit(1)


def print_table(headers: List[str], rows: List[List[Any]]) -> None:
    widths = [len(h) for h in headers]
    for row in rows:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(str(cell)))
    fmt = " | ".join(f"{{:<{w}}}" for w in widths)
    sep = "-+-".join("-" * w for w in widths)

    print(fmt.format(*headers))
    print(sep)
    for row in rows:
        print(fmt.format(*[str(c) for c in row]))
    print()


def cap_name(allcaps: Dict[str, dict], uri: str) -> str:
    c = allcaps.get(uri)
    return c.get("name") if c else uri


def human_size(n: Any) -> str:
    try:
        x = int(n)
    except Exception:
        return str(n)
    units = ["B", "KiB", "MiB", "GiB", "TiB", "PiB"]
    i = 0
    v = float(x)
    while v >= 1024.0 and i < len(units) - 1:
        v /= 1024.0
        i += 1
    if i == 0:
        return f"{int(v)} {units[i]}"
    return f"{v:.1f} {units[i]}"


def fmt_epoch(s: Any) -> str:
    """Convert epoch seconds string → UTC timestamp string."""
    try:
        e = int(str(s))
        return dt.datetime.fromtimestamp(e, tz=dt.UTC).strftime("%Y-%m-%d %H:%M:%SZ")
    except Exception:
        return str(s)


def jwt_claims(token: str) -> dict:
    """Decode JWT claims for display only. Kong and RIG verify the signature."""
    try:
        payload = token.split(".")[1]
        payload += "=" * (-len(payload) % 4)
        return json.loads(base64.urlsafe_b64decode(payload))
    except Exception:
        return {}


def error_detail(r: requests.Response) -> str:
    """One-line summary of a RIG ({error, hint}) or IRI ({detail}) error body."""
    try:
        body = r.json()
    except ValueError:
        return r.text[:300]
    if isinstance(body, dict):
        parts = [str(body[k]) for k in ("error", "title", "detail", "hint") if body.get(k)]
        if parts:
            return " | ".join(parts)[:300]
    return json.dumps(body)[:300]


def api_get(url: str, **kwargs) -> Any:
    r = requests.get(url, headers=HEADERS, timeout=TIMEOUT, **kwargs)
    if not r.ok:
        die(f"GET {url} failed: {r.status_code} {error_detail(r)}")
    return r.json()

## 2) Configuration
Loads the PAT and RIG URL written by [`login-amsc.ipynb`](login-amsc.ipynb). Environment variables (or `.env`) override them.

In [13]:
home_path = Path("~/.amsc_token.json").expanduser()
token_file = {}
if home_path.exists():
    with open(home_path) as f:
        token_file = json.load(f)

PAT = os.environ.get("AMSC_PAT")
if PAT:
    print(f"IMPORTANT: PAT defined inside .env file (or as environment variable). WILL NOT LOAD PAT from {home_path}")
else:
    PAT = token_file.get("AMSC_PAT")
if not PAT:
    die(f"Was unable to receive a PAT from AMSC_PAT or {home_path}. Use login-amsc notebook to get one.")

RIG_URL = (os.environ.get("AMSC_RIG_URL") or token_file.get("AMSC_RIG_URL") or "https://rig.staging.american-science-cloud.org").rstrip("/")
CREDENTIAL_VAULT_URL = (os.environ.get("AMSC_CREDENTIAL_VAULT_URL") or token_file.get("AMSC_CREDENTIAL_VAULT_URL") or "").rstrip("/")

claims = jwt_claims(PAT)
PROJECT = claims.get("amsc_project_context")
exp = claims.get("exp")
if exp and exp <= time.time():
    die(f"PAT expired at {fmt_epoch(exp)}. Use login-amsc notebook to get a new one.")

print("RIG         :", RIG_URL)
print("AmSC project:", PROJECT)
print("PAT expires :", fmt_epoch(exp) if exp else "unknown")

# Facility username (not your AmSC identity); used for the default job directory.
USERNAME = os.environ.get("IRI_USERNAME", "hackathon2610")

# Per-facility compute job defaults (from ../iri/compute-jobs.ipynb and ../iri/compute-jobs-alcf.ipynb).
# IRI_JOB_DIR / IRI_QUEUE / IRI_ACCOUNT override them for any facility.
# account=None means: use the facility project returned by /account/projects.
FACILITY_JOB_DEFAULTS = {
    "esnet-east": {"job_dir": "/data/home/{user}", "queue": "debug", "account": "interactive"},
    "esnet-west": {"job_dir": "/data/home/{user}", "queue": "debug", "account": "interactive"},
    "alcf": {"job_dir": "/home/{user}", "queue": "debug", "account": "AmSC_demos"},
}
GENERIC_JOB_DEFAULTS = {"job_dir": "/home/{user}", "queue": "debug", "account": None}

HEADERS = {"Authorization": f"Bearer {PAT}", "Accept": "application/json"}
POLL_INTERVAL = 5
TIMEOUT = 180

RIG         : https://rig.staging.american-science-cloud.org
AmSC project: amsc-rig-administration
PAT expires : 2026-11-07 12:01:23Z


## 3) Facility discovery through RIG
`GET <RIG>/ready` lists the facilities RIG serves (tier, API version). Each facility is then probed with your PAT,
using the same call as RIG-UI's **Test with RIG** button: the facility's `probe_path` from RIG (`/account/projects` by
default, `/account/whoami` for OLCF). This shows whether RIG can resolve a facility credential for your project.

In [14]:
def facility_api_prefix(f: dict) -> str:
    """IRI API root under the facility's RIG mount, e.g. /api/v1 (or /v1 for PNNL)."""
    metadata_path = f.get("metadata_path") or ""
    if metadata_path.endswith("/facility"):
        return metadata_path[: -len("/facility")]
    return f"/api/{f.get('api_version') or 'v1'}"


def facility_base_url(f: dict) -> str:
    return f"{RIG_URL}/rig/external/{f['name']}{facility_api_prefix(f)}"


def probe_facility(f: dict) -> tuple:
    """Make one authenticated call through RIG and return (access, detail)."""
    probe_path = f.get("probe_path") or f"{facility_api_prefix(f)}/account/projects"
    url = f"{RIG_URL}/rig/external/{f['name']}{probe_path}"
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        print(url)
    except requests.RequestException as exc:
        return "ERROR", str(exc)[:120]
    if r.ok:
        return "OK", ""
    return f"HTTP {r.status_code}", error_detail(r)[:120]


def discover_facilities() -> List[dict]:
    print("\n" + "=" * 40)
    print("=== DISCOVERING FACILITIES THROUGH RIG ===")

    r = requests.get(f"{RIG_URL}/ready", headers=HEADERS, timeout=TIMEOUT)
    if r.status_code == 401:
        die(f"RIG rejected the PAT (401): {error_detail(r)}. Use login-amsc notebook to get a new one.")
    if not r.ok:
        die(f"RIG /ready failed: {r.status_code} {error_detail(r)}")
    facilities = r.json().get("facilities", [])
    if not facilities:
        die("RIG did not return any facilities.")

    with ThreadPoolExecutor(max_workers=8) as pool:
        results = list(pool.map(probe_facility, facilities))

    rows = []
    for f, (access, detail) in zip(facilities, results):
        f["access"] = access
        f["access_detail"] = detail
        rows.append([f["name"], f.get("tier", ""), f.get("display_name") or "", facility_api_prefix(f), access, detail])

    print(f"\n=== FACILITIES (project {PROJECT}) ===")
    print_table(["Facility", "Tier", "Name", "API", "Access", "Detail"], rows)

    if any(f["access"] != "OK" for f in facilities):
        print("Some facilities are not reachable yet. See amsc-facility-access.ipynb for what each facility needs")
        print(f"(allocations, or credentials in the RIG-UI Credential Vault: {CREDENTIAL_VAULT_URL or 'see MyAmSC'}).\n")
    return facilities

## 4) Select facility
Pick the facility to run the smoke test on. `AMSC_FACILITY` preselects one; the default is the first reachable facility.

In [15]:
facilities = discover_facilities()
by_name = {f["name"]: f for f in facilities}
reachable = [f["name"] for f in facilities if f["access"] == "OK"]
if not reachable:
    print("WARNING: no facility is reachable with this PAT yet.")

default_facility = os.environ.get("AMSC_FACILITY") or (reachable[0] if reachable else "")
choice = input(f"Enter facility [{default_facility}]:\n").strip() or default_facility
if choice not in by_name:
    die(f"Unknown facility {choice!r}. Choose one of: {', '.join(by_name)}")

FACILITY = by_name[choice]
if FACILITY["access"] != "OK":
    print(f"WARNING: {choice} probe returned {FACILITY['access']} ({FACILITY['access_detail']}). Further calls will likely fail.")

BASE_URL = facility_base_url(FACILITY)
# IRI API version of the URL actually called ("v1", "v2", ...); decides the filesystem call style (step 5).
API_VERSION = facility_api_prefix(FACILITY).rstrip("/").rsplit("/", 1)[-1]

job_defaults = FACILITY_JOB_DEFAULTS.get(FACILITY["name"], GENERIC_JOB_DEFAULTS)
DEFAULT_JOB_DIR = os.environ.get("IRI_JOB_DIR", job_defaults["job_dir"].format(user=USERNAME))
DEFAULT_QUEUE = os.environ.get("IRI_QUEUE", job_defaults["queue"])
DEFAULT_ACCOUNT = os.environ.get("IRI_ACCOUNT", job_defaults["account"])

print("\nChosen facility:", FACILITY["name"], f"(tier {FACILITY.get('tier')})")
print("BASE_URL       :", BASE_URL)
print("IRI API version:", API_VERSION)


=== DISCOVERING FACILITIES THROUGH RIG ===
https://rig.staging.american-science-cloud.org/rig/external/olcf-moderate/api/v2/account/whoami
https://rig.staging.american-science-cloud.org/rig/external/esnet-east/api/v2/account/projects
https://rig.staging.american-science-cloud.org/rig/external/alcf/api/v1/account/projects
https://rig.staging.american-science-cloud.org/rig/external/olcf-open/api/v2/account/whoami
https://rig.staging.american-science-cloud.org/rig/external/nersc/api/v1/account/projects
https://rig.staging.american-science-cloud.org/rig/external/pnnl/v1/account/projects
https://rig.staging.american-science-cloud.org/rig/external/esnet-west/api/v2/account/projects

=== FACILITIES (project amsc-rig-administration) ===
Facility      | Tier | Name | API     | Access   | Detail                                                                                                                  
--------------+------+------+---------+----------+--------------------------------------

Enter facility [esnet-east]:
 esnet-east



Chosen facility: esnet-east (tier 1)
BASE_URL       : https://rig.staging.american-science-cloud.org/rig/external/esnet-east/api/v2
IRI API version: v2


## 5) Filesystem async-task helpers
Submit and poll filesystem tasks (`mkdir`, `ls`, `download`, etc.).

IRI **v1** takes `ls` / `download` as `GET` with query parameters; IRI **v2** takes them as `POST` with a JSON body.
`submit_fs_read` picks the right form from the facility's API version.

In [16]:
def submit_fs_task(method: str, path: str, **kwargs) -> dict:
    url = f"{BASE_URL}{path}"
    print(f"Submitting {method} {url}")
    r = requests.request(method, url, headers=HEADERS, timeout=TIMEOUT, **kwargs)
    if not r.ok:
        die(f"{method} {url} failed: {r.status_code} {error_detail(r)}")

    data = r.json()
    if not data.get("task_id"):
        die(f"No task_id in response: {data}")
    return data


def submit_fs_read(op: str, resource_id: str, path: str) -> dict:
    """Submit a read-style filesystem task (ls, download, ...).

    IRI v1 takes GET with query parameters; v2 takes POST with a JSON body.
    """
    if API_VERSION == "v1":
        return submit_fs_task("GET", f"/filesystem/{op}/{resource_id}", params={"path": path})
    return submit_fs_task("POST", f"/filesystem/{op}/{resource_id}", json={"path": path})


def task_url(taskin: dict) -> str:
    """URL to poll a filesystem task, always through RIG.

    A facility builds task_uri from its own view of the request. One that ignores
    RIG's X-Forwarded-Prefix returns its direct URL, which does not accept the PAT,
    so fall back to the IRI /task/{task_id} route under BASE_URL.
    """
    uri = taskin.get("task_uri") or ""
    if uri.startswith(f"{RIG_URL}/rig/external/{FACILITY['name']}/"):
        return uri
    return f"{BASE_URL}/task/{taskin['task_id']}"


def wait_fs_task(taskin: dict) -> Any:
    url = task_url(taskin)
    deadline = time.time() + TIMEOUT
    while time.time() < deadline:
        r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        if not r.ok:
            die(f"Task query failed: {r.status_code} {error_detail(r)}")

        t = r.json()
        status = t["status"]
        print(f"   FS Task {taskin['task_id']}: {status}")

        if status == "completed":
            return t.get("result")
        if status in ("failed", "canceled"):
            die(f"FS Task {taskin['task_id']} ended with status {status}: {t}")

        time.sleep(POLL_INTERVAL)

    die(f"FS Task {taskin['task_id']} timed out")

## 6) Discovery & resource selection
Lists projects/capabilities/resources on the chosen facility and prompts you to pick **compute** and **filesystem** resources.

In [17]:
def discover_resources_and_caps():
    print("\n" + "=" * 40)
    print(f"=== DISCOVERING RESOURCES AND CAPABILITIES ({FACILITY['name']}) ===")

    projects = api_get(f"{BASE_URL}/account/projects")
    project_rows = [[p.get("id"), p.get("name", ""), p.get("description", "")] for p in projects]
    print("\n=== PROJECTS ===")
    print_table(["Project ID", "Name", "Description"], project_rows)

    caps = api_get(f"{BASE_URL}/account/capabilities")
    cap_rows = [[c.get("self_uri"), c.get("name"), c.get("description", "")] for c in caps]
    cap_by_uri = {c["self_uri"]: c for c in caps}
    print("\n=== CAPABILITIES ===")
    print_table(["Capability URI", "Name", "Description"], cap_rows)

    alloc_rows = []
    project_caps = set()
    for pr in projects:
        allocs = api_get(f"{BASE_URL}/account/projects/{pr['id']}/project_allocations")
        for a in allocs:
            alloc_rows.append([pr["id"], a.get("id"), cap_name(cap_by_uri, a.get("capability_uri"))])
            if a.get("capability_uri") in cap_by_uri:
                project_caps.add(a["capability_uri"])

    print("\n=== PROJECT ALLOCATIONS ===")
    print_table(["Project ID", "Allocation ID", "Capability URI"], alloc_rows)

    if not project_caps:
        die("No allocations found for any project (need at least one capability).")

    resources = api_get(f"{BASE_URL}/status/resources", params={"offset": 0, "limit": 100})

    resource_rows = []
    for r in resources:
        rcaps = r.get("capability_uris", []) or []
        if any(cap in rcaps for cap in project_caps):
            cap_names = [cap_name(cap_by_uri, c) for c in rcaps]
            resource_rows.append(
                [
                    r.get("id"),
                    r.get("name", ""),
                    r.get("resource_type", ""),
                    r.get("description", ""),
                    ", ".join(cap_names),
                    r.get("current_status", ""),
                ]
            )

    print("\n=== RESOURCES (matching your project allocations) ===")
    print_table(
        ["Resource ID", "Name", "Type", "Description", "Capability URIs", "Current Status"],
        resource_rows,
    )

    return projects, cap_by_uri, resources, project_caps


def filter_resources(resources, cap_by_uri, project_caps, kind: str):
    kind = kind.lower().strip()
    filtered = []
    for r in resources:
        rcaps = r.get("capability_uris", []) or []
        if not any(cap in rcaps for cap in project_caps):
            continue

        rtype = (r.get("resource_type") or "").lower()
        capnames = " ".join((cap_name(cap_by_uri, c) or "").lower() for c in rcaps)

        if kind == "compute":
            if "compute" in rtype or "job" in rtype or "slurm" in rtype or "compute" in capnames:
                filtered.append(r)
        elif kind == "filesystem":
            if (
                "storage" in rtype
                or "filesystem" in rtype
                or rtype == "fs"
                or "storage" in capnames
                or "filesystem" in capnames
            ):
                filtered.append(r)
        else:
            filtered.append(r)
    return filtered


def prompt_pick_resource(resources, cap_by_uri, title: str) -> str:
    rows = []
    for r in resources:
        rcaps = r.get("capability_uris", []) or []
        capnames = ", ".join(cap_name(cap_by_uri, c) for c in rcaps)
        rows.append([r.get("id"), r.get("name", ""), r.get("resource_type", ""), capnames, r.get("current_status", "")])

    print("\n" + "=" * 40)
    print(f"=== {title} ===")
    print_table(["Resource ID", "Name", "Type", "Capabilities", "Status"], rows)

    rid = input(f"Enter {title.lower()} resource ID:\n").strip()
    if not rid:
        die("No resource ID provided.")
    return rid

## 7) Compute job submission & status polling functions
Functions to submit the compute job and poll its status until completion.

In [18]:
def submit_compute_job(compute_resource_id: str, payload: dict) -> str:
    url = f"{BASE_URL}/compute/job/{compute_resource_id}"
    print("\n" + "=" * 40)
    print(f"=== SUBMIT COMPUTE JOB: {url} ===")

    r = requests.post(url, headers={**HEADERS, "Content-Type": "application/json"}, json=payload, timeout=TIMEOUT)
    if not r.ok:
        die(f"Compute submit failed: {r.status_code} {error_detail(r)}")

    data = r.json()
    job_id = data.get("id") or data.get("job_id")
    if not job_id:
        die(f"Compute submit response missing job id: {data}")

    print(f"Compute job submitted: job_id={job_id}")
    return job_id


def poll_compute_status(compute_resource_id: str, job_id: str) -> dict:
    url = f"{BASE_URL}/compute/status/{compute_resource_id}/{job_id}"
    print("\n" + "=" * 40)
    print(f"=== POLLING COMPUTE STATUS: {url} ===")

    deadline = time.time() + TIMEOUT
    last = {}
    while time.time() < deadline:
        r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        if not r.ok:
            die(f"Compute status failed: {r.status_code} {error_detail(r)}")

        last = r.json()
        st = (last.get("status") or {})
        state = (st.get("state") or "").lower()
        msg = st.get("message", "")
        exit_code = st.get("exit_code", None)

        print(f"   job {job_id}: state={state} message={msg} exit_code={exit_code}")

        if state in {"completed", "failed", "canceled", "cancelled", "timeout", "timed_out"}:
            return last

        time.sleep(POLL_INTERVAL)

    die(f"Compute job {job_id} timed out while polling status.")

## 8) Log retrieval helpers
Pretty `ls` output + base64 decode for `download`.

In [19]:
def render_ls_table(ls_result: Any, base_path: str) -> None:
    if not isinstance(ls_result, dict):
        print("LS result (unexpected type):", ls_result)
        return

    items = ls_result.get("output")
    if not isinstance(items, list):
        print("LS result (unexpected shape):", ls_result)
        return

    def key(it):
        t = (it.get("type") or "").lower()
        is_dir = 0 if t == "directory" else 1
        return (is_dir, (it.get("name") or "").lower())

    items_sorted = sorted(items, key=key)

    rows = []
    for it in items_sorted:
        rows.append(
            [
                it.get("type", ""),
                it.get("permissions", ""),
                it.get("user", ""),
                it.get("group", ""),
                human_size(it.get("size", "")),
                fmt_epoch(it.get("last_modified", "")),
                it.get("name", ""),
                it.get("link_target") or "",
            ]
        )

    print(f"\nListing: {base_path}")
    print_table(["Type", "Perm", "UID", "GID", "Size", "Last Modified (UTC)", "Name", "Link Target"], rows)


def extract_base64_payload(download_result: Any) -> Optional[str]:
    if not isinstance(download_result, dict):
        return None

    out = download_result.get("output")

    if isinstance(out, str):
        return out

    if isinstance(out, dict):
        for k in ("content_base64", "base64", "data_base64", "data", "content"):
            v = out.get(k)
            if isinstance(v, str) and v.strip():
                return v
    return None


def decode_base64_to_text(b64: str) -> str:
    try:
        raw = base64.b64decode(b64, validate=True)
    except Exception:
        raw = base64.b64decode(b64)

    try:
        return raw.decode("utf-8")
    except UnicodeDecodeError:
        return raw.decode("utf-8", errors="replace")

## 9) Prepare for Smoke test
Identify available resources on the chosen facility and select one for the compute and filesystem

In [20]:
projects, cap_by_uri, resources, project_caps = discover_resources_and_caps()

compute_candidates = filter_resources(resources, cap_by_uri, project_caps, "compute")
fs_candidates = filter_resources(resources, cap_by_uri, project_caps, "filesystem")

if not compute_candidates:
    print("\n(No compute candidates matched heuristics; showing all matching resources instead.)")
    compute_candidates = [r for r in resources if any(c in (r.get("capability_uris") or []) for c in project_caps)]

if not fs_candidates:
    print("\n(No filesystem candidates matched heuristics; showing all matching resources instead.)")
    fs_candidates = [r for r in resources if any(c in (r.get("capability_uris") or []) for c in project_caps)]

COMPUTE_RESOURCE_ID = prompt_pick_resource(compute_candidates, cap_by_uri, "COMPUTE RESOURCE")
FS_RESOURCE_ID = prompt_pick_resource(fs_candidates, cap_by_uri, "FILESYSTEM RESOURCE")

if not DEFAULT_ACCOUNT:
    DEFAULT_ACCOUNT = projects[0]["id"] if projects else ""
    print(f"\nIRI_ACCOUNT not set: using facility project {DEFAULT_ACCOUNT!r} as the job account.")

print("\nChosen compute resource:", COMPUTE_RESOURCE_ID)
print("Chosen filesystem resource:", FS_RESOURCE_ID)
print(f"Job dir={DEFAULT_JOB_DIR} queue={DEFAULT_QUEUE} account={DEFAULT_ACCOUNT}")


=== DISCOVERING RESOURCES AND CAPABILITIES (esnet-east) ===

=== PROJECTS ===
Project ID                           | Name                   | Description                                                        
-------------------------------------+------------------------+--------------------------------------------------------------------
930784b9-161d-5750-ba65-5d2b9b3b0ea1 | Hackathon 2610 project | Compute and storage allocation for the hackathon2610 shared account


=== CAPABILITIES ===
Capability URI                                                                                                                          | Name    | Description                                      
----------------------------------------------------------------------------------------------------------------------------------------+---------+--------------------------------------------------
https://rig.staging.american-science-cloud.org/rig/external/esnet-east/api/v2/account/capabilities/7ed20f9

Enter compute resource resource ID:
 fb0aafe1-c780-55c0-b635-a7121f1b0ce5



=== FILESYSTEM RESOURCE ===
Resource ID                          | Name    | Type                                | Capabilities | Status
-------------------------------------+---------+-------------------------------------+--------------+-------
b6d59ba3-5c2d-57dd-b3b9-108832da0578 | Storage | urn:doe-iri:resource:storage:system | storage      | up    



Enter filesystem resource resource ID:
 b6d59ba3-5c2d-57dd-b3b9-108832da0578



Chosen compute resource: fb0aafe1-c780-55c0-b635-a7121f1b0ce5
Chosen filesystem resource: b6d59ba3-5c2d-57dd-b3b9-108832da0578
Job dir=/data/home/hackathon2610 queue=debug account=interactive


## 10) Submit compute job
Prepare bash script for job and submit job

In [37]:
timestamp = dt.datetime.now(dt.UTC).strftime("%Y%m%d-%H%M%S")
log_prefix = f"amsc_rig_{FACILITY['name']}_test"

job_log_name = f"{log_prefix}_{timestamp}.log"
job_log_path = f"{DEFAULT_JOB_DIR.rstrip('/')}/{job_log_name}"

stdout_path = f"{DEFAULT_JOB_DIR.rstrip('/')}/{log_prefix}_stdout_{timestamp}.log"
stderr_path = f"{DEFAULT_JOB_DIR.rstrip('/')}/{log_prefix}_stderr_{timestamp}.log"

# ========================================================
# ========================================================
# "{ echo ..." stays on one line so the script is still valid bash when it is
# joined into a single line for ALCF below.
bash_payload = f"""
set -euo pipefail
echo "=== AmSC (via RIG) compute smoke test ==="
echo "UTC now: $(date -u '+%Y-%m-%dT%H:%M:%SZ')"
echo "Hostname: $(hostname)"
echo "User: $(id || true)"
echo
echo "=== ENV (sorted) ==="
env | sort
echo
echo "=== Writing log file: {job_log_path} ==="
{{ echo "AmSC (via RIG) compute smoke test log"
  echo "UTC now: $(date -u '+%Y-%m-%dT%H:%M:%SZ')"
  echo "Hostname: $(hostname)"
  echo "---- ENV ----"
  env | sort
}} > "{job_log_path}"
echo "Wrote: {job_log_path}"
"""
# =======================================================
# =======================================================

compute_payload = {
    "executable": "bash",
    "arguments": ["-lc", bash_payload],
    "directory": DEFAULT_JOB_DIR,
    "name": f"amsc-rig-test-job-{timestamp}",
    "inherit_environment": True,
    "environment": {},
    "stdout_path": stdout_path,
    "stderr_path": stderr_path,
    "resources": {
        "node_count": 50,
        "process_count": 1,
        "processes_per_node": 1,
        "cpu_cores_per_process": 1,
        "exclusive_node_use": True,
        "memory": 268435456,
    },
    "attributes": {
        "duration": 600,
        "queue_name": DEFAULT_QUEUE,
        "account": DEFAULT_ACCOUNT,
        "custom_attributes": {},
    },
}

if FACILITY["name"] == "alcf":
    # Same adjustments as ../iri/compute-jobs-alcf.ipynb: one-line script, and fields
    # ALCF does not implement yet are left out.
    one_line = "; ".join(line.strip() for line in bash_payload.strip().splitlines() if line.strip())
    compute_payload["executable"] = "/bin/bash"
    compute_payload["arguments"] = ["-lc", one_line.replace('"', '\\"')]
    compute_payload.pop("inherit_environment")
    compute_payload.pop("environment")
    compute_payload["resources"] = {"node_count": 1, "memory": 268435456}
    compute_payload["attributes"]["custom_attributes"] = {"filesystems": "eagle"}

# ===========================================================
# SUBMIT JOB
job_id = submit_compute_job(COMPUTE_RESOURCE_ID, compute_payload)


=== SUBMIT COMPUTE JOB: https://rig.staging.american-science-cloud.org/rig/external/esnet-east/api/v2/compute/job/fb0aafe1-c780-55c0-b635-a7121f1b0ce5 ===
Compute job submitted: job_id=23412ad7-bb60-4b4d-a5c7-8c5f4f32b55d


## 11) Poll job status
Poll job status until job is finished

In [12]:
final_status = poll_compute_status(COMPUTE_RESOURCE_ID, job_id)

state = ((final_status.get("status") or {}).get("state") or "").lower()
exit_code = (final_status.get("status") or {}).get("exit_code", None)

print("\n" + "=" * 40)
print("=== COMPUTE JOB FINISHED ===")
print(f"facility : {FACILITY['name']}")
print(f"job_id   : {job_id}")
print(f"state    : {state}")
print(f"exit_code: {exit_code}")
print("=" * 40)

if state != "completed" or (exit_code not in (None, 0)):
    die(f"Compute job did not complete cleanly (state={state}, exit_code={exit_code}).")


=== POLLING COMPUTE STATUS: https://rig.staging.american-science-cloud.org/rig/external/esnet-east/api/v2/compute/status/fb0aafe1-c780-55c0-b635-a7121f1b0ce5/ded5bdc0-bddb-4608-b151-b27d0ad1a779 ===
   job ded5bdc0-bddb-4608-b151-b27d0ad1a779: state=active message=RUNNING exit_code=-1
   job ded5bdc0-bddb-4608-b151-b27d0ad1a779: state=active message=RUNNING exit_code=-1
   job ded5bdc0-bddb-4608-b151-b27d0ad1a779: state=active message=RUNNING exit_code=-1
   job ded5bdc0-bddb-4608-b151-b27d0ad1a779: state=active message=RUNNING exit_code=-1
   job ded5bdc0-bddb-4608-b151-b27d0ad1a779: state=active message=RUNNING exit_code=-1
   job ded5bdc0-bddb-4608-b151-b27d0ad1a779: state=active message=RUNNING exit_code=-1
   job ded5bdc0-bddb-4608-b151-b27d0ad1a779: state=active message=RUNNING exit_code=-1
   job ded5bdc0-bddb-4608-b151-b27d0ad1a779: state=active message=RUNNING exit_code=-1
   job ded5bdc0-bddb-4608-b151-b27d0ad1a779: state=active message=RUNNING exit_code=-1
   job ded5bdc0-b

SystemExit: 1

/Users/jbalcas/work/amsc/iri/iri-facility-api-examples/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 12) List all files and download log file

In [14]:
print("\n" + "=" * 40)
print("=== FILESYSTEM: LS JOB DIR ===")
task = submit_fs_read("ls", FS_RESOURCE_ID, DEFAULT_JOB_DIR)
ls_result = wait_fs_task(task)
render_ls_table(ls_result, DEFAULT_JOB_DIR)

print("\n" + "=" * 40)
print("=== FILESYSTEM: DOWNLOAD JOB LOG FILE ===")
task = submit_fs_read("download", FS_RESOURCE_ID, job_log_path)
download_result = wait_fs_task(task)

b64 = extract_base64_payload(download_result)
if not b64:
    die(f"Download result did not contain a base64 payload: {download_result}")

decoded = decode_base64_to_text(b64)

print("\n--- Decoded file content (UTF-8 best-effort) ---\n")
print(decoded)

local_out = f"downloaded_{job_log_name}"
with open(local_out, "w", encoding="utf-8") as f:
    f.write(decoded)
print(f"Saved: {local_out}")


=== FILESYSTEM: LS JOB DIR ===
Submitting POST https://rig.staging.american-science-cloud.org/rig/external/esnet-east/api/v2/filesystem/ls/b6d59ba3-5c2d-57dd-b3b9-108832da0578

ERROR: POST https://rig.staging.american-science-cloud.org/rig/external/esnet-east/api/v2/filesystem/ls/b6d59ba3-5c2d-57dd-b3b9-108832da0578 failed: 403 Forbidden | Filesystem operations allowed only on storage resources


SystemExit: 1